# Bloc 2.3 — File formats: CSV, JSON, Parquet

**Decision problem:** which file format preserves usability without overengineering?

Output: small format comparison and exported files.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]: d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"
def load_clean_long():
    p=OUT/"bloc1"/"clean_trends_long.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.melt(id_vars="date", var_name="signal", value_name="interest")
def load_clean_wide():
    p=OUT/"bloc1"/"clean_trends_wide.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.sort_values("date")

In [ ]:
df=load_clean_long()
base=OUT/"bloc2"
df.to_csv(base/"clean_trends_export.csv", index=False)
df.head(30).to_json(base/"clean_trends_sample.json", orient="records", indent=2, date_format="iso")
parquet_status="not_available"
try:
    df.to_parquet(base/"clean_trends_export.parquet", index=False)
    parquet_status="created"
except Exception as e:
    parquet_status=f"skipped: {type(e).__name__}"
formats=pd.DataFrame([
{"format":"CSV","best_for":"human-readable exchange","created":True},
{"format":"JSON","best_for":"API-like records and documents","created":True},
{"format":"Parquet","best_for":"analytics storage with schema/compression","created":parquet_status=="created"},
])
formats.to_csv(base/"format_comparison.csv", index=False)
print(parquet_status)
formats

## Exercise

Choose the best format for a BI dashboard and justify the trade-off.

## Conclusion

Formats are product decisions: readability, performance, schema, and interoperability.